In [1]:
import torch
print("GPU tersedia:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("Nama GPU:", torch.cuda.get_device_name(0))

GPU tersedia: True
Nama GPU: NVIDIA GeForce RTX 3060 Laptop GPU


In [9]:
import pandas as pd
import numpy as np

from datasets import Dataset
from sklearn.feature_extraction.text import TfidfVectorizer

from sklearn.svm import SVC
from sklearn.naive_bayes import MultinomialNB

from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    TrainingArguments, 
    Trainer
)

from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
import joblib
import os


In [ ]:
def check_dataset_eda():
    print("Membaca dataset chat_dataset2.csv...")
    base_dir = os.getcwd()
    dataset_path = os.path.join(base_dir, "data", "chat_dataset2.csv")
    
    if not os.path.exists(dataset_path):
        print(f"Dataset tidak ditemukan di: {dataset_path}")
        return None
        
    df = pd.read_csv(dataset_path)
    
    print("\n" + "="*40)
    print("INFORMASI UMUM DATASET")
    print("="*40)
    print(f"Total baris data mentah : {len(df)}")
    
    missing_data = df.isnull().sum().sum()
    print(f"Total data kosong (NaN) : {missing_data}")
    
    df_clean = df.dropna()
    print(f"Total data bersih       : {len(df_clean)}")
    
    print("\n" + "="*40)
    print("KOLOM DALAM DATASET")
    print("="*40)
    print(list(df.columns))
    
    print("\n" + "="*40)
    print(" JUMLAH DATA BERDASARKAN KELAS (INTENT)")
    print("="*40)
    intent_counts = df_clean['label_intent'].value_counts()
    for intent, count in intent_counts.items():
        print(f"- {intent}: {count} baris")
        
    print(f"\nTotal jenis kelas (intent): {len(intent_counts)}")
    print("="*40)
    
    return df_clean

# Jalankan fungsi EDA
df_eda = check_dataset_eda()

Membaca dataset chat_dataset_100_v2.csv...

INFORMASI UMUM DATASET
Total baris data mentah : 2794
Total data kosong (NaN) : 0
Total data bersih       : 2794

KOLOM DALAM DATASET
['teks_chat', 'label_intent']

 JUMLAH DATA BERDASARKAN KELAS (INTENT)
- claiming: 362 baris
- accusing: 360 baris
- defending: 360 baris
- bluffing: 360 baris
- persuading: 360 baris
- neutral: 360 baris
- deflecting: 330 baris
- probing: 302 baris

Total jenis kelas (intent): 8


# SVM

### SVM Before Tuning

In [12]:
def train_intent_model():
    print("Membaca dataset chat_dataset2.csv...")
    base_dir = os.getcwd()
    dataset_path = os.path.join(base_dir, "data", "chat_dataset2.csv")

    if not os.path.exists(dataset_path):
        print("Dataset tidak ditemukan! Tunggu generate.py selesai dulu ya.")
        return None

    df = pd.read_csv(dataset_path)
    df = df.dropna()

    X = df['teks_chat']
    y = df['label_intent']
   
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    print(f"Total data latih: {len(X_train)} | Total data uji: {len(X_test)}")

    tfidf = TfidfVectorizer(
        ngram_range=(1, 2),    # Mengambil kata tunggal dan pasangan 2 kata (frasa)
        min_df=2,              # Buang kata typo yang cuma muncul 1 kali di seluruh dataset
        max_df=0.9,            # Buang kata yang terlalu sering muncul (seperti "dan", "di")
        sublinear_tf=True      # Menekan dominasi kata yang di-spam berkali-kali dalam 1 chat
    )

    svm_model = SVC(
        kernel='linear',       # Linear masih oke, tapi kita tambah C
        C=2.0,                 # Coba naikkan C (bisa diubah-ubah antara 0.1, 1, 2, atau 10)
        class_weight='balanced', # Menyeimbangkan penalti jika ada kelas minoritas
        probability=True
    )

    print("Melatih model NLU (TF-IDF + SVM)...")

    model = make_pipeline(tfidf, svm_model)
    model.fit(X_train, y_train)

    print("\n--- Hasil Ujian Model (Evaluasi) ---")
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))

    os.makedirs("models", exist_ok=True)
    joblib.dump(model, "models/intent_classifier.pkl")
    print("Model berhasil disimpan di models/intent_classifier.pkl\n")

    return model

train_intent_model()

Membaca dataset chat_dataset2.csv...
Total data latih: 2235 | Total data uji: 559
Melatih model NLU (TF-IDF + SVM)...


c:\Users\andyc\Documents\a_skripsi\training\prethesis\venv\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(



--- Hasil Ujian Model (Evaluasi) ---
              precision    recall  f1-score   support

    accusing       0.67      0.78      0.72        72
    bluffing       0.59      0.64      0.61        72
    claiming       0.65      0.66      0.65        73
   defending       0.69      0.64      0.66        72
  deflecting       0.77      0.70      0.73        66
     neutral       0.94      0.89      0.91        72
  persuading       0.74      0.64      0.69        72
     probing       0.87      0.97      0.91        60

    accuracy                           0.73       559
   macro avg       0.74      0.74      0.74       559
weighted avg       0.74      0.73      0.73       559

Model berhasil disimpan di models/intent_classifier.pkl



,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidfvectorizer', ...), ('svc', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[object](8,)","['accusing','bluffing','claiming',...,'neutral','persuading','probing']"
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentn-grams to be extracted. All values of n such that min_n <= n <= max_nwill be used. For example an ``ngram_range`` of ``(1, 1)`` means onlyunigrams, ``(1, 2)`` means unigrams and bigrams, and ``(2, 2)`` meansonly bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"max_df max_df: float or int, default=1.0When building the vocabulary ignore terms that have a documentfrequency strictly higher than the given threshold (corpus-specificstop words).If float in range [0.0, 1.0], the parameter represents a proportion ofdocuments, integer absolute counts.This parameter is ignored if vocabulary is not None.",0.9
,"min_df min_df: float or int, default=1When building the vocabulary ignore terms that have a documentfrequency strictly lower than the given threshold. This value is alsocalled cut-off in the literature.If float in range of [0.0, 1.0], the parameter represents a proportionof documents, integer absolute counts.This parameter is ignored if vocabulary is not None.",2
,"sublinear_tf sublinear_tf: bool, default=FalseApply sublinear tf scaling, i.e. replace tf with 1 + log(tf).",True
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'


### SVM after tuning

In [ ]:
def train_intent_model():
    print("Membaca dataset chat_dataset2.csv...")
    base_dir = os.getcwd()
    dataset_path = os.path.join(base_dir, "data", "chat_dataset2.csv")

    if not os.path.exists(dataset_path):
        print("Dataset tidak ditemukan! Tunggu generate.py selesai dulu ya.")
        return None
    
    df = pd.read_csv(dataset_path)
    df = df.dropna()

    X = df['teks_chat']
    y = df['label_intent']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    print(f"Total data latih: {len(X_train)} | Total data uji: {len(X_test)}")
    pipeline = make_pipeline(TfidfVectorizer(), SVC(probability=True, class_weight='balanced'))

    param_grid = {
        'tfidfvectorizer__analyzer': ['word', 'char_wb'], 
        'tfidfvectorizer__ngram_range': [(1, 2), (2, 4), (1, 3)], 
        'tfidfvectorizer__use_idf': [True, False],
        
        'svc__C': [0.1, 1, 2, 5, 10],
        'svc__kernel': ['linear', 'rbf'],   
        'svc__gamma': ['scale', 'auto'],
        'svc__class_weight': ['balanced', None]
    }
    grid_search = GridSearchCV(pipeline, param_grid, cv=3, scoring='f1_weighted', verbose=2, n_jobs=-1)

    grid_search.fit(X_train, y_train)

    print(f"\nKombinasi Terbaik Ditemukan: {grid_search.best_params_}")

    best_model = grid_search.best_estimator_
    y_pred = best_model.predict(X_test)

    print("\n--- Hasil Ujian Model (Evaluasi) ---")
    print(classification_report(y_test, y_pred))

    os.makedirs("models", exist_ok=True)
    joblib.dump(best_model, "models/intent_classifier.pkl")

    return best_model

train_intent_model()

Membaca dataset chat_dataset2.csv...
Total data latih: 2235 | Total data uji: 559
Fitting 3 folds for each of 480 candidates, totalling 1440 fits


# NAIVE BAIYES

### NAIVE_BAYES Before Tuning

In [3]:
def train_intent_model_nb():
    print("Membaca dataset chat_dataset2.csv...")
    
    base_dir = os.getcwd()
    dataset_path = os.path.join(base_dir, "data", "chat_dataset2.csv")    
    model_dir = os.path.join(base_dir, "models")
    
    if not os.path.exists(dataset_path):
        print(f"Dataset tidak ditemukan di: {dataset_path}")
        return None
        
    df = pd.read_csv(dataset_path)
    df = df.dropna()
    
    X = df['teks_chat']
    y = df['label_intent']
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    print(f"Total data latih: {len(X_train)} | Total data uji: {len(X_test)}")
    
    print("Melatih model NLU (TF-IDF + Naive Bayes)...")
    model = make_pipeline(TfidfVectorizer(), MultinomialNB())
    model.fit(X_train, y_train)
    
    print("\n--- Hasil Uji Model Naive Bayes ---")
    y_pred = model.predict(X_test)
    print(classification_report(y_test, y_pred))
    
    os.makedirs("models", exist_ok=True)
    joblib.dump(model, "models/intent_classifier_nb.pkl")
    print(f"Model berhasil disimpan di models/intent_classifier_nb.pkl\n")
    
    return model

train_intent_model_nb()

Membaca dataset chat_dataset2.csv...
Total data latih: 1300 | Total data uji: 325
Melatih model NLU (TF-IDF + Naive Bayes)...

--- Hasil Uji Model Naive Bayes ---
              precision    recall  f1-score   support

    accusing       0.46      0.67      0.55        36
    bluffing       0.62      0.56      0.59        45
    claiming       0.54      0.83      0.65        36
   defending       0.72      0.59      0.65        44
  deflecting       0.70      0.56      0.62        41
     neutral       1.00      0.78      0.88        37
  persuading       0.72      0.52      0.61        44
     probing       0.83      0.93      0.88        42

    accuracy                           0.67       325
   macro avg       0.70      0.68      0.68       325
weighted avg       0.70      0.67      0.68       325

Model berhasil disimpan di models/intent_classifier_nb.pkl



,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tfidfvectorizer', ...), ('multinomialnb', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[<U10](8,)","['accusing','bluffing','claiming',...,'neutral','persuading','probing']"
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True


# TRANSFORMER

In [ ]:
def train_intent_model_transformer():
    print("Membaca dataset chat_dataset2.csv...")
    base_dir = os.getcwd()
    dataset_path = os.path.join(base_dir, "data", "chat_dataset2.csv")
        
    if not os.path.exists(dataset_path):
        print(f"Dataset tidak ditemukan di: {dataset_path}")
        return None

    df = pd.read_csv(dataset_path).dropna()
    
    label_encoder = LabelEncoder()
    df['label'] = label_encoder.fit_transform(df['label_intent'])
    num_labels = len(label_encoder.classes_)
    
    df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
    print(f"Total data latih: {len(df_train)} | Total data uji: {len(df_test)}")
    
    train_dataset = Dataset.from_pandas(df_train[['teks_chat', 'label']])
    test_dataset = Dataset.from_pandas(df_test[['teks_chat', 'label']])
    
    model_name = "indobenchmark/indobert-base-p1"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    
    def tokenize_function(examples):
        return tokenizer(examples["teks_chat"], padding="max_length", truncation=True, max_length=128)
    
    train_dataset = train_dataset.map(tokenize_function, batched=True)
    test_dataset = test_dataset.map(tokenize_function, batched=True)
    
    print("Mengunduh/Memuat model IndoBERT...")
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)
    
    training_args = TrainingArguments(
        output_dir="./models/transformer_results",
        eval_strategy="epoch",  
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=3,     
        weight_decay=0.01,
    )
    
    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        predictions = np.argmax(logits, axis=-1)
        return {"accuracy": (predictions == labels).mean()}
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        compute_metrics=compute_metrics
    )
    
    print("Mulai melatih model Transformer...")
    trainer.train()
    
    print("\n--- Hasil Uji Model Transformer ---")
    predictions = trainer.predict(test_dataset)
    y_pred = np.argmax(predictions.predictions, axis=-1)
    y_true = test_dataset["label"]
    
    target_names = label_encoder.classes_
    print(classification_report(y_true, y_pred, target_names=target_names))
    
    model_save_path = "models"
    tokenizer.save_pretrained(model_save_path)
    model.save_pretrained(model_save_path)
    
    joblib.dump(label_encoder, f"{model_save_path}/label_encoder.pkl")
    
    print(f"Model berhasil disimpan di folder: {model_save_path}\n")
    return model, tokenizer, label_encoder

train_intent_model_transformer()


Membaca dataset chat_dataset2.csv...
Total data latih: 1300 | Total data uji: 325


Map: 100%|██████████| 325/325 [00:00<00:00, 10833.09 examples/s]


Mengunduh/Memuat model IndoBERT...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at indobenchmark/indobert-base-p1 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Mulai melatih model Transformer...


  6%|▌         | 14/246 [00:08<04:30,  1.16s/it]

In [ ]:
def predict_intent(chat_text):
    model_path = "models/intent_classifier.pkl"
    if not os.path.exists(model_path):
        model = train_intent_model()
    else:
        model = joblib.load(model_path)
    
    # Prediksi intent dari chat baru
    prediksi = model.predict([chat_text])[0]
    
    # Ambil nilai probabilitas/keyakinan model (dalam persentase)
    probabilitas = max(model.predict_proba([chat_text])[0]) * 100
    
    return prediksi, probabilitas

In [ ]:
train_intent_model_transformer()
# train_intent_model_nb()
# train_intent_model()
    
# print("--- SIMULASI TESTING AI 1 DI DALAM GAME ---")
# test_chats = [
#     "Jagain rumah gw pak pol, gw bayar mahal nih pake koin",
#     "Lu curigaan mulu sama gw anjir, gw cuma warga biasa",
#     "Woy si Budi dari tadi diem aja, fix dia ketuanya"
# ]

# for chat in test_chats:
#     intent, prob = predict_intent(chat)
#     print(f"Chat Player: '{chat}'")
#     print(f" > AI 1 Menebak: [{intent.upper()}] (Tingkat Keyakinan: {prob:.2f}%)\n")